## Main question: how well can a Naive Bayes classifier distinguish letters based on numerical image features?

We know that the data in this dataset is able to be classified rather well, but the models who had their performance measure did not include Naive Bayes classification, so we'll take a look for them!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
columns = [
    "letter",
    "x_box", "y_box", "width", "height", "on_pixels",
    "x_bar", "y_bar", "x2bar", "y2bar", "xybar",
    "x2ybr", "xy2br", "x_edge", "xegvy", "y_edge", "yegvx"
]

df = pd.read_csv(
    "data/letter-recognition.data",
    header=None,
    names=columns
)
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nDataset information:")
df.info()
print("\nNumerical feature summary:")
df.describe()

In [ ]:
# Lets see if we need to do any data cleaning
print("Missing values per column:")
print(df.isnull().sum())
print("\nNumber of duplicate rows:", df.duplicated().sum())
print("\nNumber of unique letters:", df["letter"].nunique())
print("Letters present:", sorted(df["letter"].unique()))
print("\nFeature minimums:")
print(df.drop(columns="letter").min())
print("\nFeature maximums:")
print(df.drop(columns="letter").max())

# Count observations for each letter
letter_counts = df["letter"].value_counts().sort_index()
print("Observations per letter:")
print(letter_counts)
# Visualize the distribution
plt.figure(figsize=(12, 5))
sns.barplot(x=letter_counts.index, y=letter_counts.values, color="steelblue")
plt.title("Distribution of Letters in the Dataset")
plt.xlabel("Capital Letter")
plt.ylabel("Number of Observations")
plt.tight_layout()
plt.show()

In [ ]:
# Observing the numerical feature distributions
features = df.drop(columns="letter")
features.hist(
    figsize=(16, 12),
    bins=16,
    edgecolor="black",
    color="cornflowerblue"
)
plt.suptitle(
    "Distributions of Numerical Features",
    fontsize=16,
    y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate correlations between numerical features
correlation_matrix = features.corr()
plt.figure(figsize=(13, 10))
sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True
)
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:
# Inveestigating duplicate rows
feature_columns = df.columns.drop("letter")
repeated_features = df.duplicated(
    subset=feature_columns,
    keep=False
)

duplicate_groups = df.loc[repeated_features]

# Count distinct labels associated with each feature vector
label_counts_per_vector = (
    duplicate_groups.groupby(list(feature_columns))["letter"]
    .nunique()
)

# Summarize the repeated feature vectors
print("Rows belonging to repeated feature vectors:",
      repeated_features.sum())

print("Repeated feature vectors:",
      len(label_counts_per_vector))

print("Feature vectors associated with multiple letters:",
      (label_counts_per_vector > 1).sum())

# Examine exact duplicate rows with their labels
print("\nMost frequently repeated complete observations:")
print(
    df.value_counts()
    .head(10)
    .rename("frequency")
)

In [ ]:
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
strongest_correlations = (
    upper_triangle.stack()
    .rename("correlation")
    .to_frame()
)
strongest_correlations["absolute_correlation"] = (
    strongest_correlations["correlation"].abs()
)
print(
    strongest_correlations
    .sort_values("absolute_correlation", ascending=False)
    .head(10)
)

In [ ]:
# Lets go modeling!
X = df.drop(columns="letter")
y = df["letter"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the resulting dimensions
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

# Verify class proportions
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True).sort_index().round(3))

In [ ]:
# Gaussian Naive Bayes Model
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Generate predictions on unseen test rows
y_pred = gnb.predict(X_test)

# Calculate test accuracy
accuracy = accuracy_score(y_test, y_pred)
# print(f"Gaussian Naive Bayes test accuracy: {accuracy:.4f}")
print(f"Gaussian Naive Bayes test accuracy: {accuracy:.2%}")

# Detailed classification metrics
print(
    classification_report(
        y_test,
        y_pred,
        labels=sorted(y.unique()),
        zero_division=0
    )
)

In [ ]:
# Calculate the confusion matrix
labels = sorted(y.unique())

cm = confusion_matrix(y_test, y_pred, labels=labels)

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(14, 12))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

plt.title("Gaussian Naive Bayes Confusion Matrix")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Convert the confusion matrix to a DataFrame
cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)
errors = cm_df.to_numpy(copy=True)

# Exclude correct predictions along the diagonal
np.fill_diagonal(errors, 0)

# Convert back to a DataFrame
errors_df = pd.DataFrame(
    errors,
    index=labels,
    columns=labels
)

# Find the ten most frequent incorrect predictions
most_common_errors = (
    errors_df.stack()
    .sort_values(ascending=False)
    .head(10)
)

print("Ten most common misclassifications:")
print(most_common_errors)

In [ ]:
# Multinomial Naive Bayes - lets see how it performs just to check

mnb = MultinomialNB()
mnb.fit(X_train, y_train)

# Predict on the same test set
y_pred_mnb = mnb.predict(X_test)
mnb_accuracy = accuracy_score(y_test, y_pred_mnb)

# print(f"Multinomial Naive Bayes test accuracy: {mnb_accuracy:.4f}")
print(f"Multinomial Naive Bayes test accuracy: {mnb_accuracy:.2%}")

In [ ]:
# Compare test-set performance
model_comparison = pd.DataFrame({
    "Model": [
        "Gaussian Naive Bayes",
        "Multinomial Naive Bayes"
    ],
    "Test Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_mnb)
    ]
})

print(model_comparison.to_string(index=False))

# Visualize the comparison
plt.figure(figsize=(8, 5))

sns.barplot(
    data=model_comparison,
    x="Model",
    y="Test Accuracy",
    hue="Model",
    legend=False,
    palette="Set2"
)

plt.ylim(0, 1)
plt.title("Naive Bayes Model Comparison")
plt.ylabel("Test Accuracy")
plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Give identical feature vectors the same group ID
groups = pd.factorize(pd.MultiIndex.from_frame(X))[0]

# Create a stratified, group-aware split to more rigourously check it
group_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

# Verify that no feature-vector groups cross the split
train_groups = set(groups[train_idx])
test_groups = set(groups[test_idx])

print("Training observations:", len(train_idx))
print("Testing observations:", len(test_idx))
print("Shared feature-vector groups:",
      len(train_groups.intersection(test_groups)))

print("\nTraining letter distribution:")
print(y_train_group.value_counts().sort_index())

print("\nTesting letter distribution:")
print(y_test_group.value_counts().sort_index())

In [ ]:
# Train Gaussian Naive Bayes
gnb_group = GaussianNB()
gnb_group.fit(X_train_group, y_train_group)

pred_gnb_group = gnb_group.predict(X_test_group)

# Train Multinomial Naive Bayes on the same split
mnb_group = MultinomialNB()
mnb_group.fit(X_train_group, y_train_group)

pred_mnb_group = mnb_group.predict(X_test_group)

# Compare grouped-split accuracy
group_comparison = pd.DataFrame({
    "Model": [
        "Gaussian Naive Bayes",
        "Multinomial Naive Bayes"
    ],
    "Grouped Test Accuracy": [
        accuracy_score(y_test_group, pred_gnb_group),
        accuracy_score(y_test_group, pred_mnb_group)
    ]
})

print(group_comparison.to_string(index=False))

In [ ]:
# Combine the random-split and grouped-split results
evaluation_comparison = pd.DataFrame({
    "Model": [
        "Gaussian Naive Bayes",
        "Multinomial Naive Bayes"
    ],
    "Random Split Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_mnb)
    ],
    "Grouped Split Accuracy": [
        accuracy_score(y_test_group, pred_gnb_group),
        accuracy_score(y_test_group, pred_mnb_group)
    ]
})

print(evaluation_comparison.to_string(index=False))


In [ ]:
# Classification report for the grouped Gaussian model
print(
    classification_report(
        y_test_group,
        pred_gnb_group,
        labels=sorted(y.unique()),
        zero_division=0
    )
)

# Calculate grouped-split confusion matrix
cm_group = confusion_matrix(
    y_test_group,
    pred_gnb_group,
    labels=labels
)
fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_group,
    display_labels=labels
)
disp.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d"
)
plt.title("Gaussian Naive Bayes: Grouped Test Evaluation")
plt.tight_layout()
plt.show()

In [ ]:
# Build a DataFrame from the grouped confusion matrix
cm_group_df = pd.DataFrame(
    cm_group,
    index=labels,
    columns=labels
)

# Make a writable copy and exclude correct predictions
group_errors = cm_group_df.to_numpy(copy=True)
np.fill_diagonal(group_errors, 0)

group_errors_df = pd.DataFrame(
    group_errors,
    index=labels,
    columns=labels
)

# Rank incorrect actual/predicted letter pairs
top_group_errors = (
    group_errors_df.stack()
    .sort_values(ascending=False)
    .head(10)
)

print("Most common grouped-test misclassifications:")
print(top_group_errors)

In [ ]:
final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-score",
        "Weighted F1-score"
    ],
    "Gaussian Naive Bayes": [
        accuracy_score(y_test_group, pred_gnb_group),
        precision_score(
            y_test_group, pred_gnb_group,
            average="macro", zero_division=0
        ),
        recall_score(
            y_test_group, pred_gnb_group,
            average="macro", zero_division=0
        ),
        f1_score(
            y_test_group, pred_gnb_group,
            average="macro", zero_division=0
        ),
        f1_score(
            y_test_group, pred_gnb_group,
            average="weighted", zero_division=0
        )
    ]
})
print(final_metrics.to_string(index=False))

In [ ]:
# Reuse groups so identical feature vectors stay together
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate Gaussian Naive Bayes across five folds
cv_scores = cross_val_score(
    GaussianNB(),
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy",
    n_jobs=-1
)

print("Cross-validation accuracy per fold:")
print(np.round(cv_scores, 4))

print(f"\nMean accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")